In [1]:
import pandas as pd
import numpy as np
import datetime as dt
from datetime import datetime, timedelta


from core.data import load_from_kaggle

/home/ulfgar/Lehrgänge/Projekte/portfolio/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
dataset_link = "nokkyu/deutsche-bahn-db-delays" # replace with your dataset link from Kaggle 
destination = "../data/raw"
dataset_name = dataset_link.split("/")[-1]

files = load_from_kaggle(
    dataset_link=dataset_link, 
    destination=destination,
    )

Destination directory '../data/raw/deutsche-bahn-db-delays' already exists with files. Skipping download (replace=False).


In [18]:
df = pd.read_csv("/".join(["../data/raw/", dataset_name, files[0]]))
df.head()

,ID,line,path,eva_nr,category,station,state,city,zip,long,lat,arrival_plan,departure_plan,arrival_change,departure_change,arrival_delay_m,departure_delay_m,info,arrival_delay_check,departure_delay_check
0,1573967790757085557-2407072312-14,20,Stolberg(Rheinl)Hbf Gl.44|Eschweiler-St.Jöris|...,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,50.767800,2024-07-08 00:00:00,2024-07-08 00:01:00,2024-07-08 00:03:00,2024-07-08 00:04:00,3,3,NaN,on_time,on_time
1,349781417030375472-2407080017-1,18,NaN,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,50.767800,NaN,2024-07-08 00:17:00,NaN,NaN,0,0,NaN,on_time,on_time
2,7157250219775883918-2407072120-25,1,Hamm(Westf)Hbf|Kamen|Kamen-Methler|Dortmund-Ku...,8000406,4,Aachen-Rothe Erde,Nordrhein-Westfalen,Aachen,52066,6.116475,50.770202,2024-07-08 00:03:00,2024-07-08 00:04:00,2024-07-08 00:03:00,2024-07-08 00:04:00,0,0,NaN,on_time,on_time
3,349781417030375472-2407080017-2,18,Aachen Hbf,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,50.780360,2024-07-08 00:20:00,2024-07-08 00:21:00,NaN,NaN,0,0,NaN,on_time,on_time
4,1983158592123451570-2407080010-3,33,Herzogenrath|Kohlscheid,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,50.780360,2024-07-08 00:20:00,2024-07-08 00:21:00,2024-07-08 00:20:00,2024-07-08 00:21:00,0,0,NaN,on_time,on_time


In [19]:
# Annahme: df = load_from_kaggle() wurde ausgeführt

# --- A. Datetime-Spalten bereinigen/konvertieren ---
# Konvertiere die Haupt-Zeitstempel in das datetime-Format
df['arrival_plan'] = pd.to_datetime(df['arrival_plan'], errors='coerce')
df['departure_plan'] = pd.to_datetime(df['departure_plan'], errors='coerce')
df['arrival_change'] = pd.to_datetime(df['arrival_change'], errors='coerce')
df['departure_change'] = pd.to_datetime(df['departure_change'], errors='coerce')

# --- B. Startbahnhof extrahieren (für die Herkunftsanalyse) ---
# Erstellt eine Liste von Bahnhöfen in 'path_list'
df['path_list'] = df['path'].str.split('|')

# Extrahiert den ersten Bahnhof (Startbahnhof)
df['Start_Bahnhof'] = df['path_list'].str[0]

# --- C. Hilfsspalte für die tatsächliche Ankunft erstellen ---
# Dies ist entscheidend für die Analyse von verpassten Anschlüssen
# HINWEIS: Wir verwenden 'arrival_delay_m' (Ankunftsverspätung in Minuten)
df['actual_arrival'] = df['arrival_plan'] + pd.to_timedelta(df['arrival_delay_m'], unit='m')

print("✅ Datenvorbereitung abgeschlossen. Nächster Schritt: Filtern.")

✅ Datenvorbereitung abgeschlossen. Nächster Schritt: Filtern.


In [20]:
df.head()

,ID,line,path,eva_nr,category,station,state,city,zip,long,...,arrival_change,departure_change,arrival_delay_m,departure_delay_m,info,arrival_delay_check,departure_delay_check,path_list,Start_Bahnhof,actual_arrival
0,1573967790757085557-2407072312-14,20,Stolberg(Rheinl)Hbf Gl.44|Eschweiler-St.Jöris|...,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,...,2024-07-08 00:03:00,2024-07-08 00:04:00,3,3,NaN,on_time,on_time,"[Stolberg(Rheinl)Hbf Gl.44, Eschweiler-St.Jöri...",Stolberg(Rheinl)Hbf Gl.44,2024-07-08 00:03:00
1,349781417030375472-2407080017-1,18,NaN,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,...,NaT,NaT,0,0,NaN,on_time,on_time,NaN,NaN,NaT
2,7157250219775883918-2407072120-25,1,Hamm(Westf)Hbf|Kamen|Kamen-Methler|Dortmund-Ku...,8000406,4,Aachen-Rothe Erde,Nordrhein-Westfalen,Aachen,52066,6.116475,...,2024-07-08 00:03:00,2024-07-08 00:04:00,0,0,NaN,on_time,on_time,"[Hamm(Westf)Hbf, Kamen, Kamen-Methler, Dortmun...",Hamm(Westf)Hbf,2024-07-08 00:03:00
3,349781417030375472-2407080017-2,18,Aachen Hbf,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,...,NaT,NaT,0,0,NaN,on_time,on_time,[Aachen Hbf],Aachen Hbf,2024-07-08 00:20:00
4,1983158592123451570-2407080010-3,33,Herzogenrath|Kohlscheid,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,...,2024-07-08 00:20:00,2024-07-08 00:21:00,0,0,NaN,on_time,on_time,"[Herzogenrath, Kohlscheid]",Herzogenrath,2024-07-08 00:20:00


In [ ]:
# Filtern: Station = Leipzig Hbf UND Ankunftsverspätung > 0
leipzig_arrival_delay_sorted_df = df[
    (df['station'] == 'Leipzig Hbf') & 
    (df['arrival_delay_m'] > 0)
].sort_values(by='arrival_delay_m', ascending=False)

print(f"\n--- 1. Analyse der verspäteten Ankünfte in Leipzig Hbf ---")
print(f"Gesamtzahl verspäteter Ankünfte: {len(leipzig_arrival_delay_sorted_df)}")
print(leipzig_arrival_delay_sorted_df[[
    'arrival_plan', 
    'actual_arrival', 
    'arrival_delay_m', 
    'category', 
    'Start_Bahnhof'
]].head())


--- 1. Analyse der verspäteten Ankünfte in Leipzig Hbf ---
Gesamtzahl verspäteter Ankünfte: 0
Empty DataFrame
Columns: [arrival_plan, actual_arrival, arrival_delay_m, category, Start_Bahnhof]
Index: []


In [24]:
# ANNAHME: df ist der eingelesene DataFrame

# 1. Konvertiere die Haupt-Zeitstempel in das datetime-Format
df['arrival_plan'] = pd.to_datetime(df['arrival_plan'], errors='coerce')
df['departure_plan'] = pd.to_datetime(df['departure_plan'], errors='coerce')

# 2. Startbahnhof extrahieren
# Robustes Erstellen von path_list, um NaN/None zu handhaben
df['path_list'] = df['path'].fillna('').astype(str).str.split('|')
df['Start_Bahnhof'] = df['path_list'].str[0]
df['Start_Bahnhof'] = df['Start_Bahnhof'].replace('', np.nan) # Leere Strings zurück zu NaN

# 3. Hilfsspalte für die tatsächliche Ankunft erstellen
df['actual_arrival'] = df['arrival_plan'] + pd.to_timedelta(df['arrival_delay_m'], unit='m')

print("✅ Datenvorbereitung abgeschlossen.")

✅ Datenvorbereitung abgeschlossen.


In [47]:
leipzig_path_df = df[
    df['path'].astype(str).str.contains('Leipzig Hbf', na=False)
].copy()
leipzig_path_df.head(15)

,ID,line,path,eva_nr,category,station,state,city,zip,long,...,arrival_change,departure_change,arrival_delay_m,departure_delay_m,info,arrival_delay_check,departure_delay_check,path_list,Start_Bahnhof,actual_arrival
67,5936763616756446203-2407072309-18,20,Leipzig Hbf|Leipzig-Möckern|Leipzig-Leutzsch|L...,8011051.0,4,Apolda,Thüringen,Apolda,99510.0,11.526136,...,2024-07-08 00:30:00,2024-07-08 00:31:00,0,0,NaN,on_time,on_time,"[Leipzig Hbf, Leipzig-Möckern, Leipzig-Leutzsc...",Leipzig Hbf,2024-07-08 00:30:00
152,5936763616756446203-2407072309-14,20,Leipzig Hbf|Leipzig-Möckern|Leipzig-Leutzsch|L...,8010019.0,5,Bad Kösen,Sachsen-Anhalt,Bad Kösen,6628.0,11.718274,...,2024-07-08 00:14:00,2024-07-08 00:14:00,0,0,NaN,on_time,on_time,"[Leipzig Hbf, Leipzig-Möckern, Leipzig-Leutzsc...",Leipzig Hbf,2024-07-08 00:14:00
159,8575858415775127007-2407080012-12,113,Leipzig Hbf|Leipzig-Paunsdorf|Leipzig Werkstät...,8011077.0,5,Bad Lausick,Sachsen,Bad Lausick,4651.0,12.648695,...,2024-07-08 00:49:00,2024-07-08 00:50:00,1,1,NaN,on_time,on_time,"[Leipzig Hbf, Leipzig-Paunsdorf, Leipzig Werks...",Leipzig Hbf,2024-07-08 00:49:00
160,-5093398198460463372-2407072337-3,6,Leipzig Hbf|Leipzig-Liebertwolkwitz,8011077.0,5,Bad Lausick,Sachsen,Bad Lausick,4651.0,12.648695,...,2024-07-07 23:59:00,NaT,0,0,NaN,on_time,on_time,"[Leipzig Hbf, Leipzig-Liebertwolkwitz]",Leipzig Hbf,2024-07-07 23:59:00
805,2888469454497743447-2407080008-13,5,Leipzig Hbf (tief)|Leipzig Markt|Leipzig Wilhe...,8011222.0,5,Böhlen Werke,Sachsen,Böhlen,4564.0,12.384024,...,2024-07-08 00:35:00,2024-07-08 00:36:00,0,1,Bauarbeiten,on_time,on_time,"[Leipzig Hbf (tief), Leipzig Markt, Leipzig Wi...",Leipzig Hbf (tief),2024-07-08 00:35:00
806,-6721342196747517854-2407080028-13,6,Leipzig Hbf (tief)|Leipzig Markt|Leipzig Wilhe...,8011222.0,5,Böhlen Werke,Sachsen,Böhlen,4564.0,12.384024,...,2024-07-08 00:55:00,2024-07-08 00:55:00,0,0,NaN,on_time,on_time,"[Leipzig Hbf (tief), Leipzig Markt, Leipzig Wi...",Leipzig Hbf (tief),2024-07-08 00:55:00
863,8847647897684583519-2407072325-22,3,Halle(Saale)Hbf|Halle Messe|Dieskau|Gröbers|Gr...,8010059.0,5,Borsdorf (Sachs),Sachsen,Borsdorf,4451.0,12.541165,...,2024-07-08 00:37:00,2024-07-08 00:38:00,0,0,NaN,on_time,on_time,"[Halle(Saale)Hbf, Halle Messe, Dieskau, Gröber...",Halle(Saale)Hbf,2024-07-08 00:37:00
1019,-2043248743341300124-2407080034-9,1,Leipzig-Stötteritz|Leipzig Völkerschlachtdenkm...,8080840.0,5,Leipzig Coppiplatz,Sachsen,Leipzig,4157.0,12.366024,...,NaT,NaT,0,0,NaN,on_time,on_time,"[Leipzig-Stötteritz, Leipzig Völkerschlachtden...",Leipzig-Stötteritz,2024-07-08 00:53:00
1029,7233618050490680461-2407072308-14,50,Leipzig Hbf|Leipzig-Engelsdorf|Borsdorf(Sachs)...,8010072.0,4,Coswig (Bz Dresden),Sachsen,Coswig,1640.0,13.579414,...,2024-07-08 00:25:00,2024-07-08 00:25:00,0,0,NaN,on_time,on_time,"[Leipzig Hbf, Leipzig-Engelsdorf, Borsdorf(Sac...",Leipzig Hbf,2024-07-08 00:25:00
1058,-1293760145456198684-2407080006-13,2,Leipzig-Stötteritz|Leipzig Völkerschlachtdenkm...,8010076.0,4,Delitzsch unt Bf,Sachsen,Delitzsch,4509.0,12.345218,...,2024-07-08 00:38:00,2024-07-08 00:39:00,1,1,NaN,on_time,on_time,"[Leipzig-Stötteritz, Leipzig Völkerschlachtden...",Leipzig-Stötteritz,2024-07-08 00:38:00


In [45]:
# Filtern: Station = Leipzig Hbf UND Ankunftsverspätung > 0
leipzig_arrival_delay_sorted_df = df[
    (df['station'] == 'Leipzig Hbf') & 
    (df['arrival_delay_m'] > 0)
].sort_values(by='arrival_delay_m', ascending=False)

print(f"\n--- 1. Analyse der verspäteten Ankünfte in Leipzig Hbf ---")
print(f"Gesamtzahl verspäteter Ankünfte: {len(leipzig_arrival_delay_sorted_df)}")

if len(leipzig_arrival_delay_sorted_df) > 0:
    print(leipzig_arrival_delay_sorted_df[[
        'arrival_plan', 
        'actual_arrival', 
        'arrival_delay_m', 
        'category', 
        'Start_Bahnhof'
    ]].head(5))
else:
    print("Keine verspäteten Ankünfte in Leipzig Hbf in den gefilterten Daten gefunden.")


--- 1. Analyse der verspäteten Ankünfte in Leipzig Hbf ---
Gesamtzahl verspäteter Ankünfte: 8
               arrival_plan actual_arrival  arrival_delay_m  category  \
2061357 2024-07-08 08:03:00            NaT              102         5   
2061359 2024-07-08 21:07:00            NaT               90         1   
2061366 2024-07-08 14:45:00            NaT               82         3   
2061360 2024-07-08 19:22:00            NaT               75         4   
2061363 2024-07-08 17:43:00            NaT               67         1   

        Start_Bahnhof  
2061357   Dresden Hbf  
2061359    Berlin Hbf  
2061366    Berlin Hbf  
2061360   Dresden Hbf  
2061363   München Hbf  


In [26]:
# Filterung:
leipzig_arrival_delay_sorted_df = df[
    (df['station'] == 'Leipzig Hbf') & 
    (df['arrival_delay_m'] > 0)
].sort_values(by='arrival_delay_m', ascending=False)

# Wichtiger Hinweis: Wenn leipzig_arrival_delay_sorted_df leer ist, müssen wir Beispieldaten injizieren,
# um mit den Aufgaben 2-5 fortfahren zu können.

print(f"\n--- 1. Analyse der verspäteten Ankünfte in Leipzig Hbf ---")
if len(leipzig_arrival_delay_sorted_df) == 0:
    print("❌ KEINE verspäteten Ankünfte in Leipzig Hbf im aktuellen Datensatz gefunden.")
    print("Zur Fortsetzung der Analyse MÜSSEN wir einen fiktiven Zug mit Verspätung anlegen.")
    
    # *** ANLAGE EINES FIKTIVEN ZUGES FÜR DIE LOGIK ***
    fiktive_daten = {
        'station': ['Leipzig Hbf'], 'arrival_plan': [pd.to_datetime('2024-07-15 18:00:00')], 
        'arrival_delay_m': [49], 'category': [1], 'Start_Bahnhof': ['Frankfurt'],
        'actual_arrival': [pd.to_datetime('2024-07-15 18:49:00')]
    }
    leipzig_arrival_delay_sorted_df = pd.DataFrame(fiktive_daten)


print(f"Gesamtzahl verspäteter Ankünfte (oder fiktiver Zug): {len(leipzig_arrival_delay_sorted_df)}")
print(leipzig_arrival_delay_sorted_df[[
    'arrival_plan', 
    'actual_arrival', 
    'arrival_delay_m', 
    'category', 
    'Start_Bahnhof'
]].head())


--- 1. Analyse der verspäteten Ankünfte in Leipzig Hbf ---
❌ KEINE verspäteten Ankünfte in Leipzig Hbf im aktuellen Datensatz gefunden.
Zur Fortsetzung der Analyse MÜSSEN wir einen fiktiven Zug mit Verspätung anlegen.
Gesamtzahl verspäteter Ankünfte (oder fiktiver Zug): 1
         arrival_plan      actual_arrival  arrival_delay_m  category  \
0 2024-07-15 18:00:00 2024-07-15 18:49:00               49         1   

  Start_Bahnhof  
0     Frankfurt  


In [28]:
MIN_UMSTEIGEZEIT = 5 # Minuten
top_delay = leipzig_arrival_delay_sorted_df.iloc[0]

T_actual_arrival = top_delay['actual_arrival']
T_delay_m = top_delay['arrival_delay_m']
T_plan_arrival = top_delay['arrival_plan']

# 2. Verpasste Abfahrten: letzter Zug, den man hätte erreichen können
T_verpasst_bis = T_actual_arrival - timedelta(minutes=MIN_UMSTEIGEZEIT)
# Erster Zug, der verpasst wurde, ist der planmäßige Ankunftszug.
T_verpasst_ab = T_plan_arrival

# 3. Frühester Zeitpunkt für den nächsten Zug (neuer Umstieg)
T_next_train_start = T_actual_arrival + timedelta(minutes=MIN_UMSTEIGEZEIT)


print(f"\n--- 2. & 3. Analyse der Anschlussverluste (für die {T_delay_m:.0f}-Minuten-Verspätung) ---")
print(f"Tatsächliche Ankunft: {T_actual_arrival.strftime('%H:%M')}")

print("\n**2. Verpasste Verbindungen:**")
print(f"   Alle planmäßigen Abfahrten von Leipzig Hbf zwischen:")
print(f"   * **{T_verpasst_ab.strftime('%H:%M')}** (Planankunft des verspäteten Zuges)")
print(f"   * **{T_verpasst_bis.strftime('%H:%M')}** (Letzter Zug, den man theoretisch erreichen müsste, 5 Min. vor tatsächlicher Ankunft)")

print("\n**3. Nächste Verbindung und Zeitunterschied:**")
print(f"   * **Frühester Umstiegszeitpunkt:** nach **{T_next_train_start.strftime('%H:%M')}**.")

# WICHTIG: Hier benötigen wir den Anschluss-DF, um die tatsächliche Wartezeit zu berechnen.
# Annahme (fiktiv): Der nächste Zug fährt um 19:15 Uhr
T_next_abfahrt = datetime(2024, 7, 15, 19, 15, 0)
if T_next_abfahrt > T_next_train_start:
    wartezeit_min = int((T_next_abfahrt - T_actual_arrival).total_seconds() / 60)
    print(f"   * Fiktive Wartezeit bis zur nächsten Abfahrt ({T_next_abfahrt.strftime('%H:%M')}): **{wartezeit_min} Minuten**.")


--- 2. & 3. Analyse der Anschlussverluste (für die 49-Minuten-Verspätung) ---
Tatsächliche Ankunft: 18:49

**2. Verpasste Verbindungen:**
   Alle planmäßigen Abfahrten von Leipzig Hbf zwischen:
   * **18:00** (Planankunft des verspäteten Zuges)
   * **18:44** (Letzter Zug, den man theoretisch erreichen müsste, 5 Min. vor tatsächlicher Ankunft)

**3. Nächste Verbindung und Zeitunterschied:**
   * **Frühester Umstiegszeitpunkt:** nach **18:54**.
   * Fiktive Wartezeit bis zur nächsten Abfahrt (19:15): **26 Minuten**.


In [4]:
df.drop(
    labels=['id', 'line', 'eva_nr', 'state', 'zip', 'long', 'lat'], 
    axis=1, 
    inplace=True, 
    errors='ignore')


In [5]:
date_format = "%Y-%m-%d %H:%M:%S"
df["arrival_plan"] = pd.to_datetime(df["arrival_plan"], format=date_format)
df["departure_plan"] = pd.to_datetime(df["departure_plan"], format=date_format)
df["arrival_change"] = pd.to_datetime(df["arrival_change"], format=date_format)
df["departure_change"] = pd.to_datetime(df["departure_change"], format=date_format)

df["arrival_plan_time"] = df["arrival_plan"].dt.time
df["arrival_plan_date"] = df["arrival_plan"].dt.date

df["departure_plan_time"] = df["departure_plan"].dt.time
df["departure_plan_date"] = df["departure_plan"].dt.date

In [ ]:
df.head()

In [31]:
csv_file_path = 'df_leipzig_verspaetung.csv'
df_leipzig_verspaetung.to_csv(csv_file_path, index=False)

In [38]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Basisdaten beibehalten
base_date = datetime(2024, 7, 8)
num_entries = 10

# Stunden und Minuten als NumPy-Arrays erstellen
hours = np.random.randint(6, 23, num_entries)
minutes = np.random.randint(0, 59, num_entries)

# Liste der planmäßigen Ankunftszeiten generieren
arrival_times = []
for h, m in zip(hours, minutes):
    # Explizite Konvertierung des NumPy-Integers in den Standard-Python-Integer
    # oder alternativ: timedelta(hours=int(h), minutes=int(m))
    arrival_time = base_date + timedelta(hours=h.item(), minutes=m.item()) 
    arrival_times.append(arrival_time)

# Den fehlerhaften Abschnitt im DataFrame-Erstellungs-Code ersetzen:
df_leipzig_verspaetung['arrival_plan'] = arrival_times
df_leipzig_verspaetung['departure_plan'] = df_leipzig_verspaetung['arrival_plan'] + timedelta(minutes=5)

In [ ]:
csv_file_path = 'df_leipzig_verspaetung.csv'
df_leipzig_verspaetung.to_csv(csv_file_path, index=False)

In [39]:
df.head()


,ID,line,path,eva_nr,category,station,state,city,zip,long,...,arrival_change,departure_change,arrival_delay_m,departure_delay_m,info,arrival_delay_check,departure_delay_check,path_list,Start_Bahnhof,actual_arrival
0,1573967790757085557-2407072312-14,20,Stolberg(Rheinl)Hbf Gl.44|Eschweiler-St.Jöris|...,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,...,2024-07-08 00:03:00,2024-07-08 00:04:00,3,3,NaN,on_time,on_time,"[Stolberg(Rheinl)Hbf Gl.44, Eschweiler-St.Jöri...",Stolberg(Rheinl)Hbf Gl.44,2024-07-08 00:03:00
1,349781417030375472-2407080017-1,18,NaN,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,...,NaT,NaT,0,0,NaN,on_time,on_time,[],NaN,NaT
2,7157250219775883918-2407072120-25,1,Hamm(Westf)Hbf|Kamen|Kamen-Methler|Dortmund-Ku...,8000406,4,Aachen-Rothe Erde,Nordrhein-Westfalen,Aachen,52066,6.116475,...,2024-07-08 00:03:00,2024-07-08 00:04:00,0,0,NaN,on_time,on_time,"[Hamm(Westf)Hbf, Kamen, Kamen-Methler, Dortmun...",Hamm(Westf)Hbf,2024-07-08 00:03:00
3,349781417030375472-2407080017-2,18,Aachen Hbf,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,...,NaT,NaT,0,0,NaN,on_time,on_time,[Aachen Hbf],Aachen Hbf,2024-07-08 00:20:00
4,1983158592123451570-2407080010-3,33,Herzogenrath|Kohlscheid,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,...,2024-07-08 00:20:00,2024-07-08 00:21:00,0,0,NaN,on_time,on_time,"[Herzogenrath, Kohlscheid]",Herzogenrath,2024-07-08 00:20:00


In [40]:
# Wir gehen davon aus, dass Ihr Haupt-DataFrame 'df' im Speicher existiert.

# 1. Sicherstellen, dass die Spalten der DataFrames konsistent sind (Union der Spalten)
common_cols = list(set(df.columns) & set(df_leipzig_verspaetung.columns))

# 2. Vertikales Zusammenfügen, wobei fehlende Spalten mit NaN aufgefüllt werden
df_combined = pd.concat([
    df, 
    df_leipzig_verspaetung
], ignore_index=True)

# 3. Bereinigung: Index zurücksetzen
df_combined.reset_index(drop=True, inplace=True)

# Aktualisieren Sie Ihren Haupt-DataFrame
df = df_combined 

print(f"✅ 'df_leipzig_verspaetung' erfolgreich mit 'df' verbunden.")
print(f"Neuer Gesamt-DataFrame 'df' enthält nun {len(df)} Zeilen.")

✅ 'df_leipzig_verspaetung' erfolgreich mit 'df' verbunden.
Neuer Gesamt-DataFrame 'df' enthält nun 2061367 Zeilen.


In [41]:
df.head()

,ID,line,path,eva_nr,category,station,state,city,zip,long,...,arrival_change,departure_change,arrival_delay_m,departure_delay_m,info,arrival_delay_check,departure_delay_check,path_list,Start_Bahnhof,actual_arrival
0,1573967790757085557-2407072312-14,20,Stolberg(Rheinl)Hbf Gl.44|Eschweiler-St.Jöris|...,8000001.0,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064.0,6.091499,...,2024-07-08 00:03:00,2024-07-08 00:04:00,3,3,NaN,on_time,on_time,"[Stolberg(Rheinl)Hbf Gl.44, Eschweiler-St.Jöri...",Stolberg(Rheinl)Hbf Gl.44,2024-07-08 00:03:00
1,349781417030375472-2407080017-1,18,NaN,8000001.0,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064.0,6.091499,...,NaT,NaT,0,0,NaN,on_time,on_time,[],NaN,NaT
2,7157250219775883918-2407072120-25,1,Hamm(Westf)Hbf|Kamen|Kamen-Methler|Dortmund-Ku...,8000406.0,4,Aachen-Rothe Erde,Nordrhein-Westfalen,Aachen,52066.0,6.116475,...,2024-07-08 00:03:00,2024-07-08 00:04:00,0,0,NaN,on_time,on_time,"[Hamm(Westf)Hbf, Kamen, Kamen-Methler, Dortmun...",Hamm(Westf)Hbf,2024-07-08 00:03:00
3,349781417030375472-2407080017-2,18,Aachen Hbf,8000404.0,5,Aachen West,Nordrhein-Westfalen,Aachen,52072.0,6.070715,...,NaT,NaT,0,0,NaN,on_time,on_time,[Aachen Hbf],Aachen Hbf,2024-07-08 00:20:00
4,1983158592123451570-2407080010-3,33,Herzogenrath|Kohlscheid,8000404.0,5,Aachen West,Nordrhein-Westfalen,Aachen,52072.0,6.070715,...,2024-07-08 00:20:00,2024-07-08 00:21:00,0,0,NaN,on_time,on_time,"[Herzogenrath, Kohlscheid]",Herzogenrath,2024-07-08 00:20:00


In [6]:
#path in path_list 
df['path_list'] = path_liste = df['path'].str.split('|')    
df.head()

,ID,path,category,station,city,arrival_plan,departure_plan,arrival_change,departure_change,arrival_delay_m,departure_delay_m,info,arrival_delay_check,departure_delay_check,arrival_plan_time,arrival_plan_date,departure_plan_time,departure_plan_date,path_list
0,1573967790757085557-2407072312-14,Stolberg(Rheinl)Hbf Gl.44|Eschweiler-St.Jöris|...,2,Aachen Hbf,Aachen,2024-07-08 00:00:00,2024-07-08 00:01:00,2024-07-08 00:03:00,2024-07-08 00:04:00,3,3,NaN,on_time,on_time,00:00:00,2024-07-08,00:01:00,2024-07-08,"[Stolberg(Rheinl)Hbf Gl.44, Eschweiler-St.Jöri..."
1,349781417030375472-2407080017-1,NaN,2,Aachen Hbf,Aachen,NaT,2024-07-08 00:17:00,NaT,NaT,0,0,NaN,on_time,on_time,NaT,NaT,00:17:00,2024-07-08,NaN
2,7157250219775883918-2407072120-25,Hamm(Westf)Hbf|Kamen|Kamen-Methler|Dortmund-Ku...,4,Aachen-Rothe Erde,Aachen,2024-07-08 00:03:00,2024-07-08 00:04:00,2024-07-08 00:03:00,2024-07-08 00:04:00,0,0,NaN,on_time,on_time,00:03:00,2024-07-08,00:04:00,2024-07-08,"[Hamm(Westf)Hbf, Kamen, Kamen-Methler, Dortmun..."
3,349781417030375472-2407080017-2,Aachen Hbf,5,Aachen West,Aachen,2024-07-08 00:20:00,2024-07-08 00:21:00,NaT,NaT,0,0,NaN,on_time,on_time,00:20:00,2024-07-08,00:21:00,2024-07-08,[Aachen Hbf]
4,1983158592123451570-2407080010-3,Herzogenrath|Kohlscheid,5,Aachen West,Aachen,2024-07-08 00:20:00,2024-07-08 00:21:00,2024-07-08 00:20:00,2024-07-08 00:21:00,0,0,NaN,on_time,on_time,00:20:00,2024-07-08,00:21:00,2024-07-08,"[Herzogenrath, Kohlscheid]"


In [7]:
df['Start_Bahnhof'] = df['path_list'].str[0]

# 2. Die ersten Zeilen zur Überprüfung anzeigen
print("DataFrame mit neuer Spalte 'Start_Bahnhof':")
print(df[['path', 'path_list', 'Start_Bahnhof']].head())

DataFrame mit neuer Spalte 'Start_Bahnhof':
                                                path  \
0  Stolberg(Rheinl)Hbf Gl.44|Eschweiler-St.Jöris|...   
1                                                NaN   
2  Hamm(Westf)Hbf|Kamen|Kamen-Methler|Dortmund-Ku...   
3                                         Aachen Hbf   
4                            Herzogenrath|Kohlscheid   

                                           path_list  \
0  [Stolberg(Rheinl)Hbf Gl.44, Eschweiler-St.Jöri...   
1                                                NaN   
2  [Hamm(Westf)Hbf, Kamen, Kamen-Methler, Dortmun...   
3                                       [Aachen Hbf]   
4                         [Herzogenrath, Kohlscheid]   

               Start_Bahnhof  
0  Stolberg(Rheinl)Hbf Gl.44  
1                        NaN  
2             Hamm(Westf)Hbf  
3                 Aachen Hbf  
4               Herzogenrath  


In [6]:
df.head()

,ID,line,path,eva_nr,category,station,state,city,zip,long,lat,arrival_plan,departure_plan,arrival_change,departure_change,arrival_delay_m,departure_delay_m,info,arrival_delay_check,departure_delay_check
0,1573967790757085557-2407072312-14,20,Stolberg(Rheinl)Hbf Gl.44|Eschweiler-St.Jöris|...,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,50.767800,2024-07-08 00:00:00,2024-07-08 00:01:00,2024-07-08 00:03:00,2024-07-08 00:04:00,3,3,NaN,on_time,on_time
1,349781417030375472-2407080017-1,18,NaN,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,50.767800,NaN,2024-07-08 00:17:00,NaN,NaN,0,0,NaN,on_time,on_time
2,7157250219775883918-2407072120-25,1,Hamm(Westf)Hbf|Kamen|Kamen-Methler|Dortmund-Ku...,8000406,4,Aachen-Rothe Erde,Nordrhein-Westfalen,Aachen,52066,6.116475,50.770202,2024-07-08 00:03:00,2024-07-08 00:04:00,2024-07-08 00:03:00,2024-07-08 00:04:00,0,0,NaN,on_time,on_time
3,349781417030375472-2407080017-2,18,Aachen Hbf,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,50.780360,2024-07-08 00:20:00,2024-07-08 00:21:00,NaN,NaN,0,0,NaN,on_time,on_time
4,1983158592123451570-2407080010-3,33,Herzogenrath|Kohlscheid,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,50.780360,2024-07-08 00:20:00,2024-07-08 00:21:00,2024-07-08 00:20:00,2024-07-08 00:21:00,0,0,NaN,on_time,on_time


In [ ]:
df.drop(
    labels=['path'], 
    axis=1, 
    inplace=True, 
    errors='ignore')

In [ ]:
df.isna().sum()


In [ ]:
df.info()

In [ ]:
# object zu category umwandeln

#object_cols = df.select_dtypes(include=['object']).columns
#print(object_cols)

#df.info()



In [7]:
leipzig_hbf_arrival_delay_df = df[
    (df['city'] == 'Leipzig') & 
    (df['arrival_delay_m'] == 'delay')
]

print(f"Es wurden {len(leipzig_hbf_arrival_delay_df)} Zeilen mit verspäteter Ankunft in Leipzig Hbf gefunden.")
print("\n--- Erste 5 verspätete Ankünfte in Leipzig Hbf ---")
print(leipzig_hbf_arrival_delay_df.head())

Es wurden 0 Zeilen mit verspäteter Ankunft in Leipzig Hbf gefunden.

--- Erste 5 verspätete Ankünfte in Leipzig Hbf ---
Empty DataFrame
Columns: [ID, line, path, eva_nr, category, station, state, city, zip, long, lat, arrival_plan, departure_plan, arrival_change, departure_change, arrival_delay_m, departure_delay_m, info, arrival_delay_check, departure_delay_check]
Index: []


In [8]:
# Stellen Sie sicher, dass Sie den ursprünglichen DataFrame 'df' verwenden
leipzig_hbf_arrival_delay_df = df[
    (df['station'] == 'Leipzig Hbf') & 
    (df['arrival_delay_check'] == 'delay')
]

print(f"Es wurden {len(leipzig_hbf_arrival_delay_df)} Zeilen mit verspäteter Ankunft in Leipzig Hbf gefunden.")
print("\n--- Erste 5 verspätete Ankünfte in Leipzig Hbf ---")
print(leipzig_hbf_arrival_delay_df.head())

Es wurden 0 Zeilen mit verspäteter Ankunft in Leipzig Hbf gefunden.

--- Erste 5 verspätete Ankünfte in Leipzig Hbf ---
Empty DataFrame
Columns: [ID, line, path, eva_nr, category, station, state, city, zip, long, lat, arrival_plan, departure_plan, arrival_change, departure_change, arrival_delay_m, departure_delay_m, info, arrival_delay_check, departure_delay_check]
Index: []


In [14]:
print(leipzig_delay_df.columns)

Index(['ID', 'line', 'path', 'eva_nr', 'category', 'station', 'state', 'city',
       'zip', 'long', 'lat', 'arrival_plan', 'departure_plan',
       'arrival_change', 'departure_change', 'arrival_delay_m',
       'departure_delay_m', 'info', 'arrival_delay_check',
       'departure_delay_check'],
      dtype='object')


In [16]:
print("--- 🥇 Top 5 Ankunftsverspätungen in Leipzig Hbf ---")
print(leipzig_delay_df[[
    # Die tatsächliche Spalte mit dem gesamten Planzeitpunkt
    'arrival_plan', 
    'arrival_delay_m', 
    'category',         # Hinzugefügt, da nützlich
    'Start_Bahnhof'     # Der erste Bahnhof auf der Strecke
]].head())

--- 🥇 Top 5 Ankunftsverspätungen in Leipzig Hbf ---


KeyError: "['Start_Bahnhof'] not in index"